In [1]:
import pandas as pd
from pathlib import Path

In [2]:
df = pd.read_parquet(Path(r"E:\ProyectoAnalisisElectrico\PotencialesClientes\2505_2604_clustered_porcentual.parquet"))

In [3]:
df["retiro_maximo_24h"] = df.groupby("clave_unica")["medida_mean"].transform("min")
df["distancia_al_maximo"] = df["retiro_maximo_24h"] - df["medida_mean"]
df["volumen_valle_periodo"] = df.groupby("clave_unica")["distancia_al_maximo"].transform("sum") * df["medida_count"]
df["medida_total_anual"] = df["medida_total"] * df["medida_count"]/1000

In [4]:
df.head()

,clave,RUT_CLIENTE,REGION_CLIENTE,macrozona,Zona,Hora,medida_count,CLIENTE,CLIENTE_log,n_clientes,...,medida_porcentual,clave_unica,id_cluster,metodo_cluster,dtw_window,dtw_normalize,retiro_maximo_24h,distancia_al_maximo,volumen_valle_periodo,medida_total_anual
0,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,0,365,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,1,...,0.043553,$C$439_79587210-8_Antofagasta_Norte,0,kmedoids_dtw,3,False,-16659.773196,-374.163972,-9.456021e+06,-136483.591969
1,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,1,365,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,1,...,0.042765,$C$439_79587210-8_Antofagasta_Norte,0,kmedoids_dtw,3,False,-16659.773196,-668.622038,-9.456021e+06,-136483.591969
2,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,2,365,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,1,...,0.041134,$C$439_79587210-8_Antofagasta_Norte,0,kmedoids_dtw,3,False,-16659.773196,-1278.805375,-9.456021e+06,-136483.591969
3,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,3,365,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,1,...,0.042416,$C$439_79587210-8_Antofagasta_Norte,0,kmedoids_dtw,3,False,-16659.773196,-799.116594,-9.456021e+06,-136483.591969
4,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,4,365,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,1,...,0.042169,$C$439_79587210-8_Antofagasta_Norte,0,kmedoids_dtw,3,False,-16659.773196,-891.727675,-9.456021e+06,-136483.591969


In [5]:
clientes_organizados = df.groupby("clave_unica").agg({
                              "clave":"last",
                              "DEMANDA_CALOR_MWH_sum":"last", "DEMANDA_CALOR_MWH_mean":"last", "DEMANDA_CALOR_MWH_max":"last", "DEMANDA_CALOR_MWH_min":"last",
                              "volumen_valle_periodo":"last", "medida_total_anual": "last",
                              "RUT_CLIENTE":"last", "CLIENTE":"last", "CLIENTE_log":"last", "n_clientes":"last",
                              "RUBRO":"last", "SECTOR":"last", "SUBSECTOR":"last",
                              "NOMBRE_ESTABLECIMIENTO":"last", 
                              "tension":"last",
                              "REGION_CLIENTE":"last","macrozona":"last", "id_cluster":"last", 
                              "medida_count":"last","meses_operados":"last", "periodos_log":"last"})
clientes_organizados = clientes_organizados.reset_index()

In [6]:
clientes_organizados["volumen_valle_periodo"] = clientes_organizados["volumen_valle_periodo"]/1000
clientes_organizados.rename(columns={"volumen_valle_periodo":"ENERGIA_EXCEDENTE_MWH", "medida_total_anual":"ENERGIA_TOTAL_MWH"}, inplace=True)

In [7]:
energia_total = df.groupby("clave_unica")["medida_mean"].transform("sum").abs()

peak_absoluto = df.groupby("clave_unica")["medida_mean"].transform("min").abs()

horas_totales = df.groupby("clave_unica")["medida_mean"].transform("count")

factor_de_carga = energia_total / (peak_absoluto * horas_totales)
df["IndicePeakShaving"] = 1 - factor_de_carga

indice_por_cliente = df.groupby("clave_unica")["IndicePeakShaving"].last()
clientes_organizados["PuntajeForma"] = clientes_organizados["clave_unica"].map(indice_por_cliente)

clientes_organizados["PuntajeForma"] = clientes_organizados["PuntajeForma"].fillna(0)

In [8]:
clientes_organizados["PuntajeCalor"] = clientes_organizados["DEMANDA_CALOR_MWH_sum"]/clientes_organizados["DEMANDA_CALOR_MWH_sum"].max()
clientes_organizados["PuntajeExcedente"] = clientes_organizados["ENERGIA_EXCEDENTE_MWH"]/clientes_organizados["ENERGIA_EXCEDENTE_MWH"].min()
clientes_organizados["PuntajeForma"] = clientes_organizados["PuntajeForma"]/clientes_organizados["PuntajeForma"].max()
clientes_organizados["PuntajeFinal"] = (clientes_organizados["PuntajeCalor"] + clientes_organizados["PuntajeExcedente"]+ clientes_organizados["PuntajeForma"] )/3

In [14]:
# 1. Definimos los 4 trimestres con nombres descriptivos (máx 20 caracteres para que quepa "_Resumen" en Excel)
trimestres = {
    "Activos_10_a_12_Meses": [10, 11, 12],
    "Activos_7_a_9_Meses": [7, 8, 9],    
    "Activos_4_a_6_Meses": [4, 5, 6],    
    "Activos_1_a_3_Meses": [1, 2, 3],       
}

# 2. Nombre y ruta del archivo final
nombre_archivo = r"E:\ProyectoAnalisisElectrico\PotencialesClientes\Ranking_Clientes.xlsx"

# 3. Abrimos el "motor" de Excel para escribir múltiples pestañas
with pd.ExcelWriter(nombre_archivo, engine='openpyxl') as writer:
    
    for nombre_trimestre, meses in trimestres.items():
        
        # --- FILTRADO DEL TRIMESTRE ---
        # Rescatamos solo los clientes que tienen la cantidad de meses del bloque actual
        df_trim = clientes_organizados[clientes_organizados["meses_operados"].isin(meses)].copy()
        
        # Si por algún motivo no hay máquinas en este bloque, saltamos al siguiente
        if df_trim.empty:
            print(f"Aviso: No hay datos para la categoría {nombre_trimestre}.")
            continue
            
        # 1. Calculamos el puntaje estrella de cada empresa para jalar a todo el grupo hacia arriba
        df_trim["Puntaje_Max_Grupo"] = df_trim.groupby(["RUT_CLIENTE", "REGION_CLIENTE"])["PuntajeFinal"].transform("max")

        # 2. Ordenamos: Primero los mejores grupos (False), luego alfabético (True), luego el mejor fierro local (False)
        df_detalle = df_trim.sort_values(
            by=["Puntaje_Max_Grupo", "RUT_CLIENTE", "REGION_CLIENTE", "PuntajeFinal"], 
            ascending=[False, True, True, False]
        )
        
        # 3. Limpiamos la columna auxiliar del detalle para que no ensucie el Excel
        df_detalle = df_detalle.drop(columns=["Puntaje_Max_Grupo"])
        
        # --- ARMADO DE LA HOJA 'RESUMEN' ---
        # Como df_detalle ya dejó a la mejor máquina al principio de cada grupo,
        # simplemente eliminamos los duplicados por RUT+Región y conservamos el primero ("first")
        df_resumen = df_detalle.drop_duplicates(
            subset=["RUT_CLIENTE", "REGION_CLIENTE"], 
            keep="first"
        )
        
        # Ordenamos la hoja de resumen globalmente de mejor a peor para el equipo comercial
        df_resumen = df_resumen.sort_values(by="PuntajeFinal", ascending=False).reset_index(drop=True)
        
        # --- EXPORTACIÓN AL EXCEL ---
        # Escribimos las dos pestañas en nuestro archivo.
        # Nombres de hoja resultantes: "Activos_1_a_3_Meses_Resumen", etc.
        df_resumen.to_excel(writer, sheet_name=f"{nombre_trimestre}_Resumen", index=False)
        df_detalle.to_excel(writer, sheet_name=f"{nombre_trimestre}_Detalle", index=False)

print(f"✅ Archivo '{nombre_archivo}' exportado exitosamente. Revisa las pestañas en tu Excel.")

✅ Archivo 'E:\ProyectoAnalisisElectrico\PotencialesClientes\Ranking_Clientes.xlsx' exportado exitosamente. Revisa las pestañas en tu Excel.
